# CP5 · FIAP Bank · Anonimização de dados para exportação externa (LGPD)

**Aluno:** Murilo Benhossi · RM 562358 · Tecnólogo em Inteligência Artificial · RPA aplicado com IA

Pipeline completo, do CSV até a persistência das versões anonimizadas:

| Etapa | O que faz | Onde |
|---|---|---|
| a | Lê `data/manifestacoes_clientes_cp5.csv` | Seção 3 |
| b | Extrai campos estruturados via LLM com `response_schema` / Pydantic | Seções 4 e 5 |
| c | Retry com backoff, throttling, fallback Gemini → Groq, cache e idempotência | Seção 5 |
| d | Persiste texto, resumo e campos no SQLite | Seções 2 e 6 |
| e, f | Gera e persiste `texto_anonimizado` e `resumo_anonimizado` sem sobrescrever os originais | Seção 7 |
| g | Auditoria antes/depois, recall e precisão a partir de rotulagem manual | Seção 8 |
| — | Governança | Seção 9 |

## 0. Instalação
No Colab, rode a célula abaixo uma vez. Localmente, use `pip install -r requirements.txt` (ver README).

In [ ]:
import sys, subprocess, os
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/murilo2318/cp5-anonimizacao-lgpd.git"  # ajuste se o nome do repo for outro

if EM_COLAB:
    if not Path("anonimizar_texto.py").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
        os.chdir("repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "groq", "pydantic", "spacy"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download", "pt_core_news_lg"], check=True)
print("Diretorio de trabalho:", os.getcwd())

## 1. Configuração e chaves
As chaves **nunca** ficam no código. No Colab vêm do painel *Secrets* (`userdata.get`); localmente, de variáveis de ambiente `GEMINI_API_KEY` e `GROQ_API_KEY` (ou de um arquivo `.env`, que está no `.gitignore`).

In [ ]:
import os, json, time, random, hashlib, sqlite3, logging
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("cp5")


def ler_segredo(nome: str) -> str | None:
    try:
        from google.colab import userdata  # type: ignore
        valor = userdata.get(nome)
        if valor:
            return valor
    except Exception:
        pass
    if Path(".env").exists():  # leitura simples de .env local, sem dependencia extra
        for linha in Path(".env").read_text(encoding="utf-8").splitlines():
            if linha.strip().startswith(nome + "="):
                os.environ.setdefault(nome, linha.split("=", 1)[1].strip().strip('"').strip("'"))
    return os.environ.get(nome)


GEMINI_API_KEY = ler_segredo("GEMINI_API_KEY")
GROQ_API_KEY = ler_segredo("GROQ_API_KEY")
print("Gemini key carregada:", bool(GEMINI_API_KEY), "| Groq key carregada:", bool(GROQ_API_KEY))

# Modelos free-tier (confira os nomes atuais no AI Studio e no console do Groq)
MODELO_GEMINI = os.environ.get("MODELO_GEMINI", "gemini-2.5-flash")
MODELO_GROQ = os.environ.get("MODELO_GROQ", "llama-3.3-70b-versatile")

CSV_PATH = Path("data/manifestacoes_clientes_cp5.csv")
DB_PATH = Path("fiap_bank_cp5.db")

# Throttling proativo: intervalo minimo entre chamadas de cada provedor (s).
# Gemini free-tier ~10 RPM -> 6,5 s; Groq free-tier ~30 RPM -> 2,5 s.
INTERVALO_MIN = {"gemini": 6.5, "groq": 2.5}
MAX_TENTATIVAS = 4
BACKOFF_BASE = 2.0  # espera = base ** tentativa + jitter

# Demonstracao obrigatoria do fallback (secao 4a do enunciado):
# para os ids abaixo o provedor principal falha de proposito ANTES da chamada
# de rede, simulando indisponibilidade. O pipeline tem que cair no Groq.
DEMONSTRAR_FALLBACK = True
IDS_FALHA_FORCADA_GEMINI = {str(i) for i in range(1, 41) if i % 3 == 0}  # 13 de 40 ids

## 2. Banco SQLite
A tabela base é a mesma das aulas anteriores (texto, campos extraídos, resumo). As colunas novas do CP5 (`texto_anonimizado`, `resumo_anonimizado`, `provedor`) entram por **migração** com `ALTER TABLE`, só se ainda não existirem. Todas as queries são parametrizadas (`?`).

In [ ]:
def conectar(caminho: Path = DB_PATH) -> sqlite3.Connection:
    con = sqlite3.connect(caminho)
    con.row_factory = sqlite3.Row
    return con


def criar_schema(con: sqlite3.Connection) -> None:
    con.execute(
        '''CREATE TABLE IF NOT EXISTS manifestacoes (
               id                TEXT PRIMARY KEY,
               texto             TEXT NOT NULL,
               hash_texto        TEXT NOT NULL,
               tipo_manifestacao TEXT,
               sentimento        TEXT,
               valor_citado      REAL,
               urgencia          TEXT,
               resumo            TEXT,
               status            TEXT NOT NULL,          -- processado | revisao_humana | falha
               motivo_revisao    TEXT,
               processado_em     TEXT
           )'''
    )
    con.execute(
        '''CREATE TABLE IF NOT EXISTS cache_llm (
               hash_texto   TEXT PRIMARY KEY,
               provedor     TEXT NOT NULL,
               resposta     TEXT NOT NULL,
               criado_em    TEXT NOT NULL
           )'''
    )
    # Migracao CP5: colunas novas, sem apagar nada do que ja existe
    existentes = {r["name"] for r in con.execute("PRAGMA table_info(manifestacoes)")}
    for coluna in ("texto_anonimizado", "resumo_anonimizado", "provedor"):
        if coluna not in existentes:
            # nome de coluna nao aceita placeholder; vem de uma tupla fixa acima
            con.execute(f"ALTER TABLE manifestacoes ADD COLUMN {coluna} TEXT")
            log.info("Coluna adicionada: %s", coluna)
    con.commit()


con = conectar()
criar_schema(con)
pd.read_sql_query("PRAGMA table_info(manifestacoes)", con)[["name", "type"]]

## 3. Leitura do CSV

In [ ]:
df = pd.read_csv(CSV_PATH, dtype={"id": str})
print(df.shape)
df.head()

## 4. Schema de extração (Pydantic + `response_schema`)
Mesmo modelo da Aula 12. O Gemini recebe a classe no `response_schema` e devolve `resposta.parsed`; o Groq recebe o JSON Schema no prompt e a resposta é validada com o mesmo modelo.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError


class ManifestacaoExtraida(BaseModel):
    tipo_manifestacao: Literal["reclamacao", "solicitacao", "duvida", "elogio", "sugestao", "fraude"]
    sentimento: Literal["positivo", "neutro", "negativo"]
    valor_citado: Optional[float] = Field(None, description="Valor em reais citado no texto, ou null")
    urgencia: Literal["baixa", "media", "alta", "critica"]
    resumo: str = Field(..., description="Resumo objetivo em ate 25 palavras")


PROMPT_EXTRACAO = '''Voce e o robo de triagem do FIAP Bank. Leia a manifestacao do cliente e extraia:
- tipo_manifestacao: reclamacao, solicitacao, duvida, elogio, sugestao ou fraude
- sentimento: positivo, neutro ou negativo
- valor_citado: valor em reais mencionado, ou null
- urgencia: baixa, media, alta ou critica (fraude e perda de acesso sao alta/critica)
- resumo: ate 25 palavras, em portugues

Manifestacao:
<manifestacao>{texto}</manifestacao>'''

SCHEMA_JSON = json.dumps(ManifestacaoExtraida.model_json_schema(), ensure_ascii=False)

## 5. Pilha de resiliência (Aula 11)

* **Throttling proativo** (`Throttle`): garante o intervalo mínimo entre chamadas de cada provedor, antes de estourar a cota.
* **Retry com backoff exponencial + jitter** (`com_retry`): só para erros transitórios (429, 5xx, timeout, conexão). Erro permanente (chave inválida, 400) não é repetido, vai direto para o fallback.
* **Fallback Gemini → Groq** (`extrair_com_fallback`).
* **Cache** (`cache_llm`, chaveado pelo hash do texto): texto idêntico não gera nova chamada.
* **Idempotência** (`ja_processada`): consulta o banco antes de chamar a API; manifestação já persistida não é reprocessada.
* **Revisão humana**: `parsed is None` (Gemini) ou JSON fora do schema (Groq) não quebra o pipeline; a linha é gravada com `status='revisao_humana'`.

In [ ]:
from google import genai
from google.genai import types, errors as gerrors
import groq as groq_sdk

cliente_gemini = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
cliente_groq = groq_sdk.Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None


class ErroTransitorio(Exception):
    '''Vale a pena tentar de novo (cota, 5xx, rede).'''


class ErroPermanente(Exception):
    '''Nao adianta repetir (chave invalida, requisicao invalida, falha forcada).'''


class RespostaForaDoSchema(Exception):
    '''Provedor respondeu, mas sem objeto valido -> revisao humana.'''

    def __init__(self, provedor: str, trecho: str):
        super().__init__(trecho)
        self.provedor = provedor


class Throttle:
    def __init__(self, intervalos: dict[str, float]):
        self.intervalos = intervalos
        self.ultima: dict[str, float] = {}

    def aguardar(self, provedor: str) -> None:
        espera = self.intervalos[provedor] - (time.monotonic() - self.ultima.get(provedor, 0.0))
        if espera > 0:
            time.sleep(espera)
        self.ultima[provedor] = time.monotonic()


throttle = Throttle(INTERVALO_MIN)
CONTADOR_CHAMADAS = {"gemini": 0, "groq": 0}


def com_retry(func, provedor: str, *args, **kwargs):
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        throttle.aguardar(provedor)
        try:
            return func(*args, **kwargs)
        except ErroTransitorio as e:
            if tentativa == MAX_TENTATIVAS:
                raise ErroPermanente(f"{provedor}: esgotou {MAX_TENTATIVAS} tentativas ({e})") from e
            espera = BACKOFF_BASE ** tentativa + random.uniform(0, 1)
            log.warning("%s transitorio (%s). Tentativa %d/%d, aguardando %.1fs", provedor, e, tentativa, MAX_TENTATIVAS, espera)
            time.sleep(espera)


def chamar_gemini(id_: str, texto: str) -> ManifestacaoExtraida:
    if DEMONSTRAR_FALLBACK and id_ in IDS_FALHA_FORCADA_GEMINI:
        raise ErroPermanente("falha forcada do Gemini (demonstracao de fallback)")
    if cliente_gemini is None:
        raise ErroPermanente("GEMINI_API_KEY ausente")
    CONTADOR_CHAMADAS["gemini"] += 1
    try:
        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=PROMPT_EXTRACAO.format(texto=texto),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ManifestacaoExtraida,
                temperature=0,
            ),
        )
    except gerrors.APIError as e:
        if getattr(e, "code", None) in (429, 500, 502, 503, 504):
            raise ErroTransitorio(f"HTTP {e.code}") from e
        raise ErroPermanente(f"HTTP {getattr(e, 'code', '?')}: {e}") from e
    except (TimeoutError, ConnectionError) as e:
        raise ErroTransitorio(str(e)) from e

    if resposta.parsed is None:  # truncada ou fora do schema (Aula 12)
        raise RespostaForaDoSchema("gemini", (resposta.text or "")[:300])
    return resposta.parsed


def chamar_groq(id_: str, texto: str) -> ManifestacaoExtraida:
    if cliente_groq is None:
        raise ErroPermanente("GROQ_API_KEY ausente")
    CONTADOR_CHAMADAS["groq"] += 1
    try:
        resp = cliente_groq.chat.completions.create(
            model=MODELO_GROQ,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "Responda somente com um objeto JSON valido neste schema: " + SCHEMA_JSON},
                {"role": "user", "content": PROMPT_EXTRACAO.format(texto=texto)},
            ],
        )
    except (groq_sdk.RateLimitError, groq_sdk.InternalServerError, groq_sdk.APITimeoutError, groq_sdk.APIConnectionError) as e:
        raise ErroTransitorio(type(e).__name__) from e
    except groq_sdk.APIStatusError as e:
        raise ErroPermanente(f"HTTP {e.status_code}") from e

    conteudo = resp.choices[0].message.content or ""
    try:
        return ManifestacaoExtraida.model_validate_json(conteudo)
    except ValidationError as e:
        raise RespostaForaDoSchema("groq", conteudo[:300]) from e


def extrair_com_fallback(id_: str, texto: str) -> tuple[ManifestacaoExtraida, str]:
    try:
        return com_retry(chamar_gemini, "gemini", id_, texto), "gemini"
    except ErroPermanente as e:
        log.warning("id=%s Gemini indisponivel (%s). Fallback -> Groq", id_, e)
    return com_retry(chamar_groq, "groq", id_, texto), "groq"

In [ ]:
def hash_texto(texto: str) -> str:
    return hashlib.sha256(texto.strip().encode("utf-8")).hexdigest()


def ja_processada(con, id_: str) -> bool:
    '''Idempotencia: consulta o banco antes de gastar cota.'''
    linha = con.execute(
        "SELECT status FROM manifestacoes WHERE id = ? AND status IN ('processado', 'revisao_humana')", (id_,)
    ).fetchone()
    return linha is not None


def buscar_cache(con, h: str):
    linha = con.execute("SELECT provedor, resposta FROM cache_llm WHERE hash_texto = ?", (h,)).fetchone()
    if linha:
        return ManifestacaoExtraida.model_validate_json(linha["resposta"]), linha["provedor"]
    return None


def gravar_cache(con, h: str, dados: ManifestacaoExtraida, provedor: str) -> None:
    con.execute(
        "INSERT OR IGNORE INTO cache_llm (hash_texto, provedor, resposta, criado_em) VALUES (?, ?, ?, ?)",
        (h, provedor, dados.model_dump_json(), datetime.now(timezone.utc).isoformat()),
    )


def gravar_manifestacao(con, id_, texto, h, dados, provedor, status, motivo=None) -> None:
    d = dados.model_dump() if dados else {}
    con.execute(
        '''INSERT INTO manifestacoes
               (id, texto, hash_texto, tipo_manifestacao, sentimento, valor_citado, urgencia, resumo,
                status, motivo_revisao, provedor, processado_em)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
           ON CONFLICT(id) DO UPDATE SET
               tipo_manifestacao = excluded.tipo_manifestacao, sentimento = excluded.sentimento,
               valor_citado = excluded.valor_citado, urgencia = excluded.urgencia, resumo = excluded.resumo,
               status = excluded.status, motivo_revisao = excluded.motivo_revisao,
               provedor = excluded.provedor, processado_em = excluded.processado_em''',
        (id_, texto, h, d.get("tipo_manifestacao"), d.get("sentimento"), d.get("valor_citado"),
         d.get("urgencia"), d.get("resumo"), status, motivo, provedor,
         datetime.now(timezone.utc).isoformat()),
    )


def processar_lote(con, df: pd.DataFrame) -> dict:
    stats = {"pulados_idempotencia": 0, "cache": 0, "gemini": 0, "groq": 0, "revisao_humana": 0, "falha": 0}
    for id_, texto in zip(df["id"], df["texto"]):
        if ja_processada(con, id_):
            stats["pulados_idempotencia"] += 1
            continue
        h = hash_texto(texto)
        em_cache = buscar_cache(con, h)
        if em_cache:
            dados, provedor = em_cache
            gravar_manifestacao(con, id_, texto, h, dados, provedor, "processado")
            stats["cache"] += 1
        else:
            try:
                dados, provedor = extrair_com_fallback(id_, texto)
                gravar_cache(con, h, dados, provedor)
                gravar_manifestacao(con, id_, texto, h, dados, provedor, "processado")
                stats[provedor] += 1
            except RespostaForaDoSchema as e:
                log.error("id=%s resposta fora do schema -> revisao humana", id_)
                gravar_manifestacao(con, id_, texto, h, None, e.provedor, "revisao_humana", f"parsed is None: {e}")
                stats["revisao_humana"] += 1
            except ErroPermanente as e:
                log.error("id=%s falhou nos dois provedores: %s", id_, e)
                gravar_manifestacao(con, id_, texto, h, None, None, "falha", str(e))
                stats["falha"] += 1
        con.commit()  # commit por linha: se cair no meio, o que ja foi pago fica salvo
    return stats

### 5.1 Primeira execução (com falha forçada do Gemini em parte do lote)

In [ ]:
inicio = time.time()
stats_1 = processar_lote(con, df)
print(stats_1, "| chamadas de API:", CONTADOR_CHAMADAS, f"| {time.time() - inicio:.0f}s")

### 5.2 Evidência do fallback no próprio banco
A coluna `provedor` precisa ter uma mistura real de `gemini` e `groq`.

In [ ]:
pd.read_sql_query(
    "SELECT provedor, COUNT(*) AS qtd FROM manifestacoes GROUP BY provedor ORDER BY provedor", con
)

In [ ]:
# Os ids em que o Gemini foi derrubado devem aparecer com provedor = 'groq'
placeholders = ",".join("?" * len(IDS_FALHA_FORCADA_GEMINI))
pd.read_sql_query(
    f"SELECT id, provedor, status FROM manifestacoes WHERE id IN ({placeholders}) ORDER BY CAST(id AS INTEGER)",
    con, params=sorted(IDS_FALHA_FORCADA_GEMINI, key=int),
)

### 5.3 Segunda execução: idempotência
Rodar de novo sobre o mesmo CSV não pode duplicar linhas nem chamar a API.

In [ ]:
antes = dict(CONTADOR_CHAMADAS)
stats_2 = processar_lote(con, df)
print(stats_2)
print("Chamadas novas de API:", {k: CONTADOR_CHAMADAS[k] - antes[k] for k in antes})
print("Linhas na tabela:", con.execute("SELECT COUNT(*) FROM manifestacoes").fetchone()[0], "| linhas no CSV:", len(df))
assert all(CONTADOR_CHAMADAS[k] == antes[k] for k in antes), "segunda execucao gastou cota"
assert con.execute("SELECT COUNT(*) FROM manifestacoes").fetchone()[0] == df["id"].nunique()

### 5.4 Fila de revisão humana
Linhas em que o LLM não devolveu objeto válido (`parsed is None`) ou em que os dois provedores falharam.

In [ ]:
pd.read_sql_query(
    "SELECT id, status, motivo_revisao FROM manifestacoes WHERE status <> 'processado' ORDER BY CAST(id AS INTEGER)", con
)

## 6. Campos extraídos persistidos

In [ ]:
pd.read_sql_query(
    '''SELECT id, tipo_manifestacao, sentimento, valor_citado, urgencia, provedor, resumo
       FROM manifestacoes ORDER BY CAST(id AS INTEGER)''', con
)

## 7. Anonimização local (spaCy + regex)

A função está no arquivo `anonimizar_texto.py`, na raiz do repositório, com a assinatura fixa `anonimizar_texto(texto: str) -> str`. Ela trata os dois problemas como complementares:

* **Identificadores estruturados** (CPF, RG, conta/agência, e-mail): expressões regulares por formato e, quando o formato é ambíguo, pela palavra-chave anterior (`RG 501987654` vira `[RG]`; `conta 1234` vira `[CONTA]`; `cartão final 4521` fica intacto).
* **Nomes de pessoa**: NER do spaCy (`PER`), com filtro de falsos positivos (verbos no início de frase que o modelo marca como pessoa, como “Fui”, “Achei”) e gatilhos de contexto (“meu nome é”, “gerente”, “atendente”, “Att,”, “Assinado,”) para recuperar nomes em minúsculas ou rotulados como LOC/ORG.

Aplicada em `texto` e em `resumo`, gravando em colunas próprias. As colunas originais não são alteradas. A etapa também é idempotente: só processa linhas com `texto_anonimizado IS NULL`.

In [ ]:
import importlib
import anonimizar_texto as anon
importlib.reload(anon)
from anonimizar_texto import anonimizar_texto

print("Modelo spaCy em uso:", anon._carregar_spacy().meta["name"] if anon._carregar_spacy() else "nenhum (fallback regex)")

pendentes = con.execute(
    "SELECT id, texto, resumo FROM manifestacoes WHERE texto_anonimizado IS NULL"
).fetchall()
for linha in pendentes:
    con.execute(
        "UPDATE manifestacoes SET texto_anonimizado = ?, resumo_anonimizado = ? WHERE id = ?",
        (anonimizar_texto(linha["texto"]),
         anonimizar_texto(linha["resumo"]) if linha["resumo"] else None,
         linha["id"]),
    )
con.commit()
print(f"{len(pendentes)} linhas anonimizadas nesta execucao")

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.read_sql_query(
    '''SELECT id, texto, texto_anonimizado, resumo, resumo_anonimizado
       FROM manifestacoes WHERE texto <> texto_anonimizado ORDER BY CAST(id AS INTEGER)''', con
)

### 7.1 Visão de exportação
O que sai para a consultoria: só colunas anonimizadas e campos sem PII. O texto original não entra na exportação.

In [ ]:
exportacao = pd.read_sql_query(
    '''SELECT id, tipo_manifestacao, sentimento, valor_citado, urgencia,
              texto_anonimizado AS texto, resumo_anonimizado AS resumo
       FROM manifestacoes WHERE status = 'processado' ORDER BY CAST(id AS INTEGER)''', con
)
exportacao.to_csv("exportacao_consultoria.csv", index=False)
exportacao.head(10)

## 8. Auditoria

### 8.1 Rotulagem manual

O dataset não tem gabarito. A rotulagem está em `rotulagem_manual.json`: cada id rotulado traz a lista de PII que existe no texto, com o tipo e o trecho exato. Os 40 ids foram rotulados (o mínimo pedido é 15). Ids **sem** PII ficam com lista vazia: eles entram na precisão, porque a função não pode mascarar texto comum.

Critérios usados na rotulagem (seção 2 do enunciado):

* Nome de qualquer pessoa citada, inclusive atendente, gerente e primeiro nome isolado (“A cliente Camila”).
* A palavra-chave antes do número (“CPF”, “conta”, “agência”) **não** é PII; só o número.
* Não são PII nesta atividade: final de cartão com 4 dígitos, valores, datas, telefone, nomes de produto (Pix) e cidades.

### 8.2 Métrica

Avaliação caixa-preta: usa só a saída de `anonimizar_texto`, como o teste do professor. Texto original e saída são tokenizados e alinhados com `difflib.SequenceMatcher`. Token do original que aparece num bloco igual do alinhamento foi preservado; o resto foi removido ou mascarado.

* **Recall** = tokens de PII que sumiram da saída ÷ total de tokens de PII rotulados.
* **Precisão** (definição do enunciado) = tokens não-PII que ficaram intactos ÷ total de tokens não-PII.

In [ ]:
import difflib, re

_TOKEN = re.compile(r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)+|\w+(?:[.\-/]\w+)*|[^\w\s]", re.UNICODE)


def tokenizar(texto: str):
    return [(m.group(0), m.start(), m.end()) for m in _TOKEN.finditer(texto)]


def avaliar_texto(texto: str, saida: str, rotulos: list[dict]) -> dict:
    tok_orig = tokenizar(texto)
    tok_out = [t for t, _, _ in tokenizar(saida)]
    sm = difflib.SequenceMatcher(a=[t for t, _, _ in tok_orig], b=tok_out, autojunk=False)
    preservado = [False] * len(tok_orig)
    for bloco in sm.get_matching_blocks():
        for k in range(bloco.a, bloco.a + bloco.size):
            preservado[k] = True

    spans = []
    for r in rotulos:
        ini = texto.find(r["texto"])
        if ini < 0:
            raise ValueError(f"rotulo nao encontrado: {r['texto']!r}")
        spans.append((ini, ini + len(r["texto"]), r["tipo"]))

    res = {"pii_total": 0, "pii_removido": 0, "nao_pii_total": 0, "nao_pii_intacto": 0, "fn": [], "fp": []}
    for (tok, a, b), ok in zip(tok_orig, preservado):
        tipo = next((t for (i, f, t) in spans if a >= i and b <= f), None)
        if tipo:
            res["pii_total"] += 1
            res["pii_removido"] += (not ok)
            if ok:
                res["fn"].append(f"{tok} ({tipo})")
        else:
            res["nao_pii_total"] += 1
            res["nao_pii_intacto"] += ok
            if not ok:
                res["fp"].append(tok)
    return res


def avaliar_conjunto(textos: dict, rotulos: dict, funcao) -> tuple[pd.DataFrame, float, float]:
    linhas = []
    for id_, rot in rotulos.items():
        saida = funcao(textos[id_])
        r = avaliar_texto(textos[id_], saida, rot)
        linhas.append({"id": id_, "saida": saida, **r})
    tab = pd.DataFrame(linhas)
    recall = tab["pii_removido"].sum() / tab["pii_total"].sum()
    precisao = tab["nao_pii_intacto"].sum() / tab["nao_pii_total"].sum()
    return tab, recall, precisao


rotulagem = json.loads(Path("rotulagem_manual.json").read_text(encoding="utf-8"))["rotulos"]
textos = dict(zip(df["id"], df["texto"]))
tab_texto, recall_texto, precisao_texto = avaliar_conjunto(textos, rotulagem, anonimizar_texto)
print(f"Campo TEXTO | ids rotulados: {len(rotulagem)} | tokens PII: {tab_texto.pii_total.sum()} "
      f"| recall: {recall_texto:.3f} | precisao: {precisao_texto:.3f}")
tab_texto[["id", "pii_total", "pii_removido", "fn", "fp", "saida"]]

### 8.3 Auditoria do resumo gerado pela IA

O resumo pode reintroduzir um nome ou número que estava no texto. Duas checagens automáticas por id, usando a rotulagem do texto como referência:

* **Falso negativo no resumo:** algum trecho de PII rotulado no texto (ou parte de nome com 3+ letras, sem diferenciar maiúsculas) sobrevive em `resumo_anonimizado`.
* **Falso positivo no resumo:** aparece um marcador de um tipo que não existe na rotulagem daquele id (ex.: `[NOME]` num resumo de manifestação sem nome). Esses casos precisam de leitura manual para confirmar.

Os resumos mudam a cada execução do LLM, então esta tabela precisa ser lida **depois** da execução que gerou o `.db` entregue.

In [ ]:
def pii_que_sobrou(resumo_anon: str, rotulos: list[dict]) -> list[str]:
    if not isinstance(resumo_anon, str) or not resumo_anon:  # NaN/None: revisao humana
        return []
    baixo = resumo_anon.lower()
    sobras = []
    for r in rotulos:
        partes = [r["texto"]] + ([p for p in re.findall(r"[^\W\d_]+", r["texto"]) if len(p) >= 3 and p.lower() not in {"das", "dos"}]
                                 if r["tipo"] == "NOME" else [re.sub(r"\D", "", r["texto"])])
        for p in partes:
            if p and (p.lower() in baixo or (r["tipo"] != "NOME" and p in re.sub(r"\D", "", resumo_anon) and len(p) >= 4)):
                sobras.append(f"{p} ({r['tipo']})")
                break
    return sobras


def marcadores_suspeitos(resumo_anon: str, rotulos: list[dict]) -> list[str]:
    if not isinstance(resumo_anon, str) or not resumo_anon:  # NaN/None: revisao humana
        return []
    tipos_ok = {r["tipo"] for r in rotulos}
    return [m for m in re.findall(r"\[(NOME|CPF|RG|CONTA|EMAIL)\]", resumo_anon) if m not in tipos_ok]


resumos = pd.read_sql_query("SELECT id, resumo, resumo_anonimizado FROM manifestacoes", con).set_index("id")
aud_resumo = []
for id_, rot in rotulagem.items():
    if id_ in resumos.index:
        ra = resumos.loc[id_, "resumo_anonimizado"]
        aud_resumo.append({"id": id_, "resumo": resumos.loc[id_, "resumo"], "resumo_anonimizado": ra,
                           "pii_sobrando": pii_que_sobrou(ra, rot),
                           "marcador_suspeito": marcadores_suspeitos(ra, rot)})
aud_resumo = pd.DataFrame(aud_resumo)
print("Resumos com PII sobrando:", (aud_resumo["pii_sobrando"].str.len() > 0).sum(),
      "| resumos com marcador suspeito:", (aud_resumo["marcador_suspeito"].str.len() > 0).sum())
aud_resumo[aud_resumo["resumo"] != aud_resumo["resumo_anonimizado"]]

### 8.4 Amostras antes/depois

Amostra fixa (seed) para leitura manual, incluindo ids sem PII, onde qualquer alteração seria falso positivo.

In [ ]:
amostra = pd.read_sql_query(
    "SELECT id, texto, texto_anonimizado, resumo, resumo_anonimizado FROM manifestacoes", con
).sample(10, random_state=42).sort_values("id", key=lambda s: s.astype(int))
amostra

### 8.5 Geração do `auditoria_resultado.json`

Os casos de erro vêm da execução real: primeiro os falsos negativos/positivos no texto, depois os do resumo. A célula não inventa casos. Se houver menos de 2, ela avisa; nesse caso a saída é ler os resumos e as amostras com atenção e registrar o que for encontrado, nunca inventar um caso.

**Revise as descrições antes do commit final**: elas são geradas a partir dos tokens e precisam ser lidas e, se necessário, reescritas com o que de fato aconteceu naquele id.

In [ ]:
casos = []
for _, r in tab_texto.iterrows():
    if r["fn"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_negativo",
                      "descricao": f"No campo texto, a funcao deixou passar {', '.join(r['fn'])}. Saida: {r['saida']}"})
    if r["fp"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_positivo",
                      "descricao": f"No campo texto, a funcao mascarou por engano {', '.join(r['fp'])}. Saida: {r['saida']}"})
for _, r in aud_resumo.iterrows():
    if r["pii_sobrando"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_negativo",
                      "descricao": f"No resumo gerado pela IA, sobrou {', '.join(r['pii_sobrando'])}. "
                                   f"Resumo anonimizado: {r['resumo_anonimizado']}"})
    if r["marcador_suspeito"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_positivo",
                      "descricao": f"No resumo gerado pela IA, a funcao inseriu {', '.join('[' + m + ']' for m in r['marcador_suspeito'])} "
                                   f"num id sem esse tipo de PII (confirmar lendo o resumo). "
                                   f"Resumo: {r['resumo']} -> {r['resumo_anonimizado']}"})

resultado = {
    "recall_estimado": round(float(recall_texto), 2),
    "precisao_estimado": round(float(precisao_texto), 2),
    "casos_de_erro": casos,
}
if len(casos) < 2:
    print(f"ATENCAO: so {len(casos)} caso(s) real(is) de erro nesta execucao. Nao invente casos: "
          "leia os resumos (8.3) e as amostras (8.4) e registre no relatorio (8.6) o que foi encontrado.")
Path("auditoria_resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

### 8.6 Relatório de auditoria

> **Preencher depois da execução final**, comentando os mesmos ids que estão em `casos_de_erro` do JSON. Para cada um: o que havia no texto, o que a função fez, por que errou (regra, limitação do NER, formato não previsto) e se o erro é de recall (vazou PII) ou de precisão (mascarou texto comum).

**Leitura dos números.** O recall e a precisão acima foram medidos nos ids rotulados do conjunto público. As regras de contexto da função (“meu nome é”, “Att,”, palavra-chave antes de número) foram escritas olhando esse mesmo conjunto, então o número no público é otimista em relação a um conjunto novo. Nos testes com frases fora do CSV, os erros que apareceram foram:

* nome em minúsculas sem gatilho antes (“falei com joana ontem”): o spaCy não marca como pessoa e nenhuma regra de contexto dispara, porque a função não tem gatilho para esse caso;
* inicial solta depois do nome (“Silvia R.”): o nome é mascarado, a inicial fica;
* apelidos e nomes que também são palavras comuns (“Seu Jorge”, “Rosa”) dependem do modelo acertar a entidade.

**Por que o modelo importa.** O `pt_core_news_sm` rotula alguns nomes completos como LOC ou ORG (ex.: nome japonês, nome terminado em sobrenome pouco comum). A função compensa isso com gatilhos de contexto, mas sem gatilho o erro aparece. Por isso o README pede o `pt_core_news_lg`.

**Casos do JSON.** _(descrever aqui, id por id)_

## 9. Governança

### 9.1 Por que anonimizar localmente com spaCy e não com mais uma chamada de LLM

> **Confira os três motivos com o que foi discutido na Aula 13 e ajuste a redação se o professor usou outros termos.**

1. **Exposição do dado.** Mandar o texto original para um LLM externo para “remover a PII” é, em si, um novo compartilhamento de dado pessoal com um terceiro, geralmente com servidores fora do Brasil (transferência internacional, art. 33 da LGPD). Seria resolver a exportação para a consultoria criando outra exportação, justamente o problema que se quer evitar. Com spaCy e regex, o texto não sai da máquina do FIAP Bank.
2. **Previsibilidade e auditoria.** Regex e NER local dão sempre a mesma saída para a mesma entrada, podem ser testados com casos fixos e medidos (recall/precisão, seção 8). Um LLM varia entre execuções, pode resumir ou reescrever o texto em vez de só mascarar, e pode deixar passar ou até reintroduzir um nome sem aviso. Numa etapa de conformidade, precisamos conseguir explicar por que cada trecho foi mascarado.
3. **Custo, cota e disponibilidade.** A anonimização roda sobre todo o histórico, a cada exportação. Localmente não consome cota free-tier, não depende de rede nem do provedor estar no ar, e não passa pelo throttling. A pilha de resiliência da seção 5 existe porque as APIs falham; a etapa de LGPD não deve depender delas.

### 9.2 Por que o dado original continua no banco

* **Atendimento.** A manifestação precisa ser respondida ao cliente certo. Sem nome, CPF ou conta, o banco não consegue localizar a transação contestada, estornar valor ou tratar uma fraude.
* **Auditoria e prestação de contas.** A LGPD pede que o controlador registre as operações de tratamento (art. 37) e consiga demonstrar as medidas adotadas (art. 6º, X). A própria auditoria da seção 8 só é possível comparando o original com a versão anonimizada.
* **Direitos do titular.** O cliente pode pedir acesso, correção ou informação sobre o uso dos seus dados (art. 18). Isso exige o dado identificado.
* **Anonimização é irreversível e pode errar.** Se a função tiver um erro (seção 8), o original permite corrigir a função e gerar de novo a versão anonimizada. Se só existisse a versão mascarada, o erro ficaria permanente, ou pior, a PII que vazou não teria como ser localizada.

A separação é por finalidade: o original fica no banco interno com acesso restrito, e só a visão de exportação (seção 7.1), com as colunas anonimizadas, sai para a consultoria. Dado anonimizado deixa de ser dado pessoal (art. 12) justamente porque não carrega o vínculo com o titular; o banco interno continua sendo dado pessoal e continua sujeito a todas as obrigações da LGPD.

In [ ]:
con.close()